In [ ]:
import pandas as pd

df = pd.read_csv("../../data/processed/hotel_general_info_geopy_fixed.csv")

In [19]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import pandas as pd

geolocator = Nominatim(
    user_agent="hotel_geocoder_vn",
    timeout=10   # ⬅️ tăng từ mặc định (1s) lên 10s
)
geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1.5,
    max_retries=2,
    error_wait_seconds=5,
    swallow_exceptions=True
)

def geocode_address(address):
    try:
        location = geocode(address, language="vi")
        if location:
            return location.latitude, location.longitude
    except Exception:
        pass
    return None, None



def shorten_address(addr):
    if pd.isna(addr) or addr.strip() == "":
        return None

    parts = [p.strip() for p in addr.split(",")]

    keep = []
    for p in parts:
        pl = p.lower()

        if any(k in pl for k in ["đường", "đại lộ", "street"]):
            keep.append(p)
        elif any(k in pl for k in ["phường", "quận", "thành phố", "tỉnh"]):
            keep.append(p)
        elif "vietnam" in pl:
            keep.append("Vietnam")

    return ", ".join(dict.fromkeys(keep))



In [20]:
# df = df[:10].copy()
df["geopy_address"] = df["hotel_address"].apply(shorten_address)
df[["lat", "lng"]] = df["geopy_address"].apply(
    lambda x: pd.Series(geocode_address(x))
)

In [21]:
total_rows = len(df)
print("Tổng số dòng:", total_rows)

success_count = df["lat"].notna() & df["lng"].notna()
success_count = success_count.sum()
print("Số dòng có lat/lng:", success_count)
fail_count = total_rows - success_count
print("Số dòng bị NaN:", fail_count)
print(f"Tỉ lệ thành công: {success_count / total_rows:.2%}")
print(f"Tỉ lệ thất bại: {fail_count / total_rows:.2%}")

Tổng số dòng: 4271
Số dòng có lat/lng: 3609
Số dòng bị NaN: 662
Tỉ lệ thành công: 84.50%
Tỉ lệ thất bại: 15.50%


In [22]:
# pd.set_option("display.max_colwidth", None)
# df["hotel_address"]